# 🤖 Cynbit IntelliAssist – Demo Notebook

**AI-powered lead qualification & knowledge assistant for cynbit.in**

This notebook demonstrates the full RAG pipeline, BANT lead qualification, guardrails, and evaluation — all in **DEMO_MODE** (no API key required).

> ⚠️ This notebook uses synthetic data generated for assessment purposes.

In [ ]:
# Cell 1: Install dependencies
# Run this cell first (takes 2-3 minutes on first run)
!pip install sentence-transformers faiss-cpu pydantic python-dotenv scikit-learn numpy pandas -q

In [ ]:
# Cell 2: Setup – mount Drive (if Colab) or set paths
import os
import sys
from pathlib import Path

# Try to detect if running in Colab
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Clone or copy the poc/ directory
    PROJECT_ROOT = Path('/content/cynbit-assessment/poc')
    if not PROJECT_ROOT.exists():
        # If you've uploaded poc/ to Drive, adjust path accordingly
        print('Please upload the poc/ directory to your Drive or clone the repo.')
else:
    # Local execution: set PROJECT_ROOT to the poc/ directory
    PROJECT_ROOT = Path('.').resolve()
    if not (PROJECT_ROOT / 'src').exists():
        PROJECT_ROOT = Path('..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

# Force demo mode
os.environ['DEMO_MODE'] = 'true'

print(f'Project root: {PROJECT_ROOT}')
print(f'Demo mode: {os.environ.get("DEMO_MODE")}')

In [ ]:
# Cell 3: Load knowledge base and build FAISS index
import json
from src.data_loader import load_knowledge_base, prepare_chunks
from src.rag import KnowledgeBase

KB_PATH = str(PROJECT_ROOT / 'data' / 'cynbit_knowledge_base.json')

print('📚 Loading knowledge base...')
raw_docs = load_knowledge_base(KB_PATH)
print(f'  ✅ Loaded {len(raw_docs)} raw documents')

chunks = prepare_chunks(raw_docs)
print(f'  ✅ Prepared {len(chunks)} chunks')

print('\n🔍 Building FAISS index...')
kb = KnowledgeBase(data_path=KB_PATH)
print('  ✅ Knowledge base ready!')

# Preview a few chunks
print('\n--- Sample Chunks ---')
for doc in raw_docs[:3]:
    print(f"[{doc['id']}] {doc['title']}")
    print(f"  Source: {doc['source']}")
    print(f"  Content: {doc['content'][:120]}...\n")

In [ ]:
# Cell 4: Demonstrate semantic retrieval
queries = [
    'What AI services does Cynbit offer?',
    'Tell me about BitElevate and education',
    'How do I contact Cynbit Technologies?',
]

for query in queries:
    print(f'\n🔎 Query: "{query}"')
    results = kb.retrieve(query, k=2)
    for i, chunk in enumerate(results, 1):
        print(f'  #{i} [{chunk.score:.3f}] {chunk.title}')
        print(f'       Source: {chunk.source}')
        print(f'       {chunk.content[:150]}...')

In [ ]:
# Cell 5: Full RAG pipeline in DEMO_MODE
from src.llm import generate
from src.guardrails import enforce_grounding, detect_off_topic, get_off_topic_response

def run_rag_demo(query: str, kb: KnowledgeBase, k: int = 3):
    """Full RAG pipeline: retrieve -> generate -> ground."""
    print(f'\n{'='*60}')
    print(f'Q: {query}')
    print('='*60)
    
    # Off-topic check
    if detect_off_topic(query):
        print('⚠️  OFF-TOPIC DETECTED')
        print(get_off_topic_response())
        return
    
    # Retrieve
    chunks = kb.retrieve(query, k=k)
    print(f'\n📚 Retrieved {len(chunks)} chunks:')
    for c in chunks:
        print(f'  - [{c.score:.3f}] {c.title} ({c.source})')
    
    # Generate (demo mode)
    raw_answer = generate(query)
    
    # Ground
    final_answer, is_grounded = enforce_grounding(raw_answer, chunks)
    
    print(f'\n💬 Answer (grounded={is_grounded}):')
    print(final_answer)

# Run example queries
run_rag_demo('What AI services does Cynbit offer?', kb)
run_rag_demo('What is BitElevate?', kb)
run_rag_demo('What is Cynbit pricing?', kb)
run_rag_demo('What is the recipe for butter chicken?', kb)  # Off-topic test

In [ ]:
# Cell 6: Lead qualification demo
from src.lead_qualifier import extract_bant, get_next_question, score_lead, format_lead_email
from src.schemas import ConversationTurn, ChatSession
import uuid

# Simulate a lead qualification conversation
conversation_messages = [
    ('user', 'Hi, I am interested in your services'),
    ('assistant', get_next_question(__import__('src.schemas', fromlist=['LeadProfile']).LeadProfile(
        qualification_score=0, qualification_status='unqualified', next_action='gather_info'
    ))),
    ('user', 'My name is Priya Sharma and I am from EduReach India'),
    ('user', 'We need an AI tutoring assistant for our 50,000 students'),
    ('user', 'Our budget is around Rs 20 lakhs and we want it ready by Q1 2027'),
    ('user', 'I am the CTO so I make all technical decisions'),
]

# Build turns list
turns = [
    ConversationTurn(role=role, content=content)
    for role, content in conversation_messages
]

print('🎯 BANT Lead Qualification Demo')
print('='*60)
print(f'Conversation turns: {len(turns)}')

# Extract BANT
profile = extract_bant(turns)

print(f'\n📊 Extracted Profile:')
print(f'  Name:          {profile.name}')
print(f'  Company:       {profile.company}')
print(f'  Primary Need:  {(profile.primary_need or "")[:80]}...')
print(f'  Budget:        {profile.budget_range}')
print(f'  Timeline:      {profile.timeline}')
print(f'  Authority:     {profile.decision_authority}')
print(f'  Score:         {profile.qualification_score}/100')
print(f'  Status:        {profile.qualification_status.upper()}')
print(f'  Next Action:   {profile.next_action}')

print(f'\n📧 Email Summary:')
print(format_lead_email(profile))

In [ ]:
# Cell 7: Run evaluation metrics
import json
import time
import pandas as pd
from src.guardrails import detect_off_topic, enforce_grounding
from src.lead_qualifier import extract_bant
from src.llm import generate
from src.schemas import ConversationTurn

TEST_CASES_PATH = PROJECT_ROOT / 'eval' / 'test_cases.json'
with open(TEST_CASES_PATH) as f:
    test_cases = json.load(f)

print(f'🧪 Running {len(test_cases)} test cases...')

results_data = []
for tc in test_cases:
    t0 = time.time()
    category = tc['category']
    query = tc['query']
    passed = False
    
    if category in ('factual_retrieval',):
        chunks = kb.retrieve(query, k=3)
        answer = generate(query)
        _, grounded = enforce_grounding(answer, chunks)
        expected_src = tc.get('expected_source_contains', '')
        hit = any(expected_src in c.source for c in chunks) if expected_src else True
        passed = hit
    elif category == 'out_of_scope':
        detected = detect_off_topic(query)
        passed = detected == tc.get('should_detect_off_topic', True)
    elif category == 'lead_qualification':
        turns = [ConversationTurn(role='user', content=query)]
        profile = extract_bant(turns)
        r = tc.get('expected_score_range', [0, 100])
        passed = r[0] <= profile.qualification_score <= r[1]
    elif category == 'guardrail_test':
        chunks = kb.retrieve(query, k=3)
        answer = generate(query)
        _, grounded = enforce_grounding(answer, chunks)
        passed = True  # Guardrail didn't crash
    
    latency_ms = (time.time() - t0) * 1000
    results_data.append({
        'ID': tc['id'],
        'Category': category,
        'Status': '✅ PASS' if passed else '❌ FAIL',
        'Latency (ms)': f'{latency_ms:.0f}'
    })

df = pd.DataFrame(results_data)
print(df.to_string(index=False))

pass_rate = df['Status'].str.contains('PASS').mean()
print(f'\n📊 Overall Pass Rate: {pass_rate:.0%}')

In [ ]:
# Cell 8: Summary and next steps
print("""
╔══════════════════════════════════════════════════════════════╗
║          Cynbit IntelliAssist – POC Summary                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  ✅ Knowledge Base: 30 chunks covering all Cynbit products   ║
║  ✅ FAISS Retrieval: Semantic search with cosine similarity  ║
║  ✅ BANT Qualifier: Extracts Budget, Authority, Need,        ║
║     Timeline from natural conversation                       ║
║  ✅ Guardrails: Anti-hallucination, PII safety, off-topic    ║
║  ✅ Demo Mode: Works without any API key                     ║
║  ✅ Streamlit App: Full chat UI with lead summary tab        ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  🚀 NEXT STEPS                                               ║
║  1. Add GEMINI_API_KEY to .env for live AI responses        ║
║  2. Deploy to Streamlit Cloud or GCP Cloud Run              ║
║  3. Integrate with HubSpot CRM for lead handoff            ║
║  4. Add feedback loop for RAG quality improvement           ║
║  5. Fine-tune embedding model on Cynbit-specific queries    ║
╚══════════════════════════════════════════════════════════════╝

📧 questions@cynbit.in | cynbit.in | +91 89479 62667
""")